In [ ]:
# !git clone https://github.com/torchcvnn/examples

fatal: destination path 'examples' already exists and is not an empty directory.


In [39]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import seaborn as sns
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from tqdm import tqdm
import h5py
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import complextorch.nn.modules.conv as CVConv
import complextorch.nn.modules.linear as CVLinear
import complextorch.nn.modules.pooling as CVPooling
import complextorch.nn.modules.activation as CVActivation
import complextorch.nn.modules.batchnorm as CVBatchNorm

/home/shane/miniconda3/envs/cvnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# !pip install torchcvnn
import torchcvnn.nn as c_nn

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Check torchcvnn.nn for names like ModReLU, zReLU, Cardioid, etc.

In [5]:
def get_complex_activation(name: str):
    name = name.lower()
    if name == "modrelu":
        return c_nn.modReLU()
    elif name == "zrelu":
        return c_nn.zReLU()
    elif name == "cardioid":
        return c_nn.Cardioid()
    elif name == "c_relu":
        return c_nn.CReLU()
    elif name == "c_sigmoid":
        return c_nn.CSigmoid()
    elif name == "c_tanh":
        return c_nn.CTanh()
    elif name == "c_elu":
        return c_nn.CELU()
    elif name == "c_gelu":
        return c_nn.CGELU()
    else:
        return nn.Identity()

In [6]:
class ComplexMnistCNN(nn.Module):
    def __init__(self, act_name="modrelu"):
        super().__init__()
        act = get_complex_activation(act_name)

        self.features = nn.Sequential(
            c_nn.ConvTranspose2d(1, 16, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(16),
            act,
            c_nn.ConvTranspose2d(16, 32, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(32),
            act,
            c_nn.AvgPool2d(kernel_size=2, stride=2),  
            c_nn.ConvTranspose2d(32, 64, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(64),
            act,
            c_nn.AvgPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128, dtype=torch.complex64),
            act,
            nn.Linear(128, 10, dtype=torch.complex64),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        # For classification, map complex logits to real via magnitude or real part.
        return x.abs()

In [7]:
class RealMnistCNN(nn.Module):
    def __init__(self, use_two_channels=False):
        super().__init__()
        in_ch = 2 if use_two_channels else 1

        self.features = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.ConvTranspose2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
class ComplexFourierMNIST(torch.utils.data.Dataset):
    def __init__(self, mnist_dataset):
        self.base = mnist_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]        # img: [1,28,28], real
        img = torch.fft.fft2(img)          # convert to complex
        img = img.to(torch.complex64)
        return img, label

Training Loop

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, epoch, act_name):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc=f"[{act_name}] Train {epoch}", leave=False)
    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)            # real-valued logits from complex net
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        _, preds = logits.max(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[{act_name}] Epoch {epoch} | loss={avg_loss:.4f} | acc={acc:.4f}")

In [10]:
def real_train_one_epoch(real_model, loader, optimizer, criterion, epoch):
    real_model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc=f"[Real CNN Train {epoch}", leave=False)
    for x, y in pbar:
        xr = torch.view_as_real(x)            # [B,1,28,28,2]
        xr = xr.squeeze(1)                    # [B,28,28,2]
        xr = xr.permute(0, 3, 1, 2)           # [B,2,28,28]
        xr = xr.float().to(device)

        y = y.to(device)

        
        optimizer.zero_grad()

        logits = real_model(xr)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        _, preds = logits.max(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[Real CNN] Epoch {epoch} | loss={avg_loss:.4f} | acc={acc:.4f}")

Eval Loop

In [11]:
def evaluate(model, loader, criterion, act_name):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        pbar = tqdm(loader, desc=f"[{act_name}] Eval", leave=False)
        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            _, preds = logits.max(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[{act_name}] Val | loss={avg_loss:.4f} | acc={acc:.4f}")
    return avg_loss, acc

In [13]:
def real_evaluate(real_model, loader, criterion):
    real_model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        pbar = tqdm(loader, desc=f"[Real CNN Eval", leave=False)
        for x, y in pbar:
            xr = torch.view_as_real(x)        # [B,1,28,28,2]
            xr = xr.squeeze(1)                # [B,28,28,2]
            xr = xr.permute(0, 3, 1, 2)       # [B,2,28,28]
            xr = xr.float().to(device)

            y = y.to(device)

            logits = real_model(xr)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            _, preds = logits.max(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[Real CNN] | loss={avg_loss:.4f} | acc={acc:.4f}")
    return avg_loss, acc

Main

In [12]:
transform = T.Compose([
    T.ToTensor(),  # [0,1] float, shape [1,28,28]
])
train_real = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_real  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [13]:
train_ds = ComplexFourierMNIST(train_real)
test_ds  = ComplexFourierMNIST(test_real)

In [14]:
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

CVNN

In [13]:
activations_to_test = ["modrelu", "zrelu", "cardioid", "c_relu", "c_sigmoid", "c_tanh", "c_elu", "c_gelu"]

for act_name in activations_to_test:
    print(f"\n=== Testing activation: {act_name} ===")
    model = ComplexMnistCNN(act_name=act_name).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, 6):
        train_one_epoch(model, train_loader, optimizer, criterion, epoch, act_name)
        evaluate(model, test_loader, criterion, act_name)



=== Testing activation: modrelu ===


[modrelu] Epoch 1 | loss=0.3020 | acc=0.9118


[modrelu] Val | loss=0.2029 | acc=0.9389


[modrelu] Epoch 2 | loss=0.1608 | acc=0.9528


[modrelu] Val | loss=0.1572 | acc=0.9501


...

[c_gelu] Val | loss=0.0467 | acc=0.9838


[c_gelu] Epoch 4 | loss=0.0230 | acc=0.9926


[c_gelu] Val | loss=0.0471 | acc=0.9852


[c_gelu] Epoch 5 | loss=0.0153 | acc=0.9950


[c_gelu] Val | loss=0.0453 | acc=0.9871


In [15]:
x0, y0 = next(iter(train_loader))
print("dataset batch dtype:", x0.dtype, x0.shape)


dataset batch dtype: torch.complex64 torch.Size([128, 1, 28, 28])


In [16]:
print(dir(c_nn))

['AdaptiveAvgPool2d', 'AvgPool2d', 'BatchNorm1d', 'BatchNorm2d', 'CCELU', 'CELU', 'CGELU', 'CPReLU', 'CReLU', 'CSigmoid', 'CTanh', 'Cardioid', 'ComplexMSELoss', 'ConvTranspose2d', 'Dropout', 'Dropout2d', 'LayerNorm', 'MaxPool2d', 'Mod', 'MultiheadAttention', 'RMSNorm', 'Transformer', 'TransformerDecoderLayer', 'TransformerEncoder', 'TransformerEncoderLayer', 'Upsample', 'ViT', 'ViTLayer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'activation', 'batchnorm', 'conv', 'dropout', 'functional', 'init', 'initialization', 'loss', 'modReLU', 'modules', 'normalization', 'pooling', 'transformer', 'upsampling', 'vit', 'zAbsReLU', 'zLeakyReLU', 'zReLU']


Real NN

In [21]:
real_model = RealMnistCNN(use_two_channels=True).to(device)
optimizer = optim.Adam(real_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 6):
    real_train_one_epoch(real_model, train_loader, optimizer, criterion, epoch )
    real_evaluate(real_model, test_loader, criterion)


[Real CNN] Epoch 1 | loss=0.1690 | acc=0.9485


[Real CNN] | loss=0.0894 | acc=0.9718


[Real CNN] Epoch 2 | loss=0.0566 | acc=0.9821


[Real CNN] | loss=0.0798 | acc=0.9738


[Real CNN] Epoch 3 | loss=0.0411 | acc=0.9871


[Real CNN] | loss=0.0697 | acc=0.9771


[Real CNN] Epoch 4 | loss=0.0325 | acc=0.9895


[Real CNN] | loss=0.0579 | acc=0.9805


[Real CNN] Epoch 5 | loss=0.0251 | acc=0.9914


[Real CNN] | loss=0.0517 | acc=0.9832


# RADIOML

In [41]:
data_loc = "/mnt/i/RADIOML/"

In [40]:
with h5py.File(data_loc + "GOLD_XYZ_OSC.0001_1024.hdf5", "r") as f:
    print(list(f.keys()))   
    x = f["X"][:1000]
    y = f["Y"][:1000]
    z = f["Z"][:1000]
    print(x.shape, y.shape, z.shape)


['X', 'Y', 'Z']
(1000, 1024, 2) (1000, 24) (1000, 1)


We have N frames, with 1024 time-series samples, and they are real and imaginuary, we have 2. so X will be N x 1024 x 2.

Y is the labels, one-hot encoded so we have 24 labels, only 1 of them should have 1.

Z SNR - signal to noise ratio and each series has an associated value

In [79]:
class ComplexRadioCNN(nn.Module):
    def __init__(self, n_classes=24):
        super().__init__()
        self.features = nn.Sequential(
            CVConv.Conv1d(1, 16, kernel_size=7, padding=3),
            # CVBatchNorm.BatchNorm1d(16),
            CVActivation.modReLU(bias=-0.1),      # <-- set bias < 0
            CVConv.Conv1d(16, 32, kernel_size=7, padding=3),
            # CVBatchNorm.BatchNorm1d(32),
            CVActivation.modReLU(bias=-0.1),
            CVPooling.AdaptiveAvgPool1d(512),   # 1024 -> 512
            CVConv.Conv1d(32, 64, kernel_size=7, padding=3),
            # CVBatchNorm.BatchNorm1d(64),
            CVActivation.modReLU(bias=-0.1),
            CVPooling.AdaptiveAvgPool1d(256),   # 512 -> 256
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            CVLinear.Linear(64 * 256, 256),
            CVActivation.modReLU(bias=-0.1),
            CVLinear.Linear(256, n_classes),
        )

    def forward(self, z):
        z = self.features(z)
        z = self.classifier(z)
        return z.abs()


In [80]:
X = x.astype("float32")
X_complex = X[..., 0] + 1j * X[..., 1]        # shape (N, 1024), complex64
X_complex = torch.from_numpy(X_complex).to(torch.complex64)  # (N, 1024)
X_complex = X_complex.unsqueeze(1)            # (N, 1, 1024) for Conv1d-style nets

In [81]:
X_real = torch.from_numpy(X).permute(0, 2, 1).float()  # (N, 2, 1024)


In [82]:
Y_idx = y.argmax(axis=1)          # numpy, shape (N,)
Y_idx = torch.from_numpy(Y_idx).long()

In [83]:
class RadioDataset(Dataset):
    def __init__(self, X_complex, Y_idx):
        self.X = X_complex   # [N,1,1024], complex
        self.y = Y_idx       # [N], long

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]

In [84]:
N = X_complex.shape[0]
split = int(0.8 * N)
train_ds = RadioDataset(X_complex[:split], Y_idx[:split])
test_ds  = RadioDataset(X_complex[split:], Y_idx[split:])

In [85]:

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

In [86]:
def train_one_epoch(model, loader, optimizer, criterion, epoch, tag="Complex"):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, desc=f"[{tag}] Train {epoch}", leave=False):
        x = x.to(device)      # complex
        y = y.to(device)      # long

        optimizer.zero_grad()
        logits = model(x)     # [B, n_classes], real
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        _, preds = logits.max(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[{tag}] Epoch {epoch} | loss={avg_loss:.4f} | acc={acc:.4f}")

In [87]:
def evaluate(model, loader, criterion, tag="Complex"):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in tqdm(loader, desc=f"[{tag}] Eval", leave=False):
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            _, preds = logits.max(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[{tag}] Val | loss={avg_loss:.4f} | acc={acc:.4f}")
    return avg_loss, acc

In [88]:
model = ComplexRadioCNN(n_classes=24).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 6):
    train_one_epoch(model, train_loader, optimizer, criterion, epoch, tag="CVNN")
    evaluate(model, test_loader, criterion, tag="CVNN")

[CVNN] Epoch 1 | loss=1.3513 | acc=0.8400


[CVNN] Val | loss=0.0239 | acc=1.0000


[CVNN] Epoch 2 | loss=0.0010 | acc=1.0000


[CVNN] Val | loss=0.0113 | acc=0.9950


[CVNN] Epoch 3 | loss=0.0030 | acc=0.9988


[CVNN] Val | loss=0.0174 | acc=0.9900


[CVNN] Epoch 4 | loss=0.0001 | acc=1.0000


[CVNN] Val | loss=0.0671 | acc=0.9950


[CVNN] Epoch 5 | loss=0.0389 | acc=0.9950


[CVNN] Val | loss=0.4205 | acc=0.9800


In [ ]:
# print([n for n in dir(CVPooling)])

['AdaptiveAvgPool1d', 'AdaptiveAvgPool2d', 'AdaptiveAvgPool3d', 'Tuple', 'Union', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'cvF', 'nn', 'torch']


## CPLXModule

In [ ]:
# !pip install cplxmodule
from cplxmodule import cplx
from cplxmodule.nn import CplxConv1d, CplxLinear, CplxModReLU

class CplxRadioCNN(nn.Module):
    def __init__(self, n_classes=24):
        super().__init__()
        self.features = nn.Sequential(
            CplxConv1d(1, 16, kernel_size=7, padding=3),
            CplxModReLU(),
            CplxConv1d(16, 32, kernel_size=7, padding=3),
            CplxModReLU(),
            cplx.AvgPool1d(kernel_size=2),   # simple wrapper around real pool
            CplxConv1d(32, 64, kernel_size=7, padding=3),
            CplxModReLU(),
            cplx.AvgPool1d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            cplx.Flatten(),
            CplxLinear(64 * 256, 256),
            CplxModReLU(),
            CplxLinear(256, n_classes),
        )

    def forward(self, z):
        # z: complex tensor [B,1,1024] (torch.complex64)
        z = self.features(z)
        z = self.classifier(z)
        return abs(z)   # real logits [B, n_classes]